In [ ]:
import seaborn as sns

In [ ]:
titanic_df = sns.load_dataset('titanic')
titanic_df.info()

In [ ]:
# 컬럼 의미
print("""
survived : 생존 여부 (0=사망, 1=생존) ← 예측 대상(Target)
pclass   : 객실 등급 (1=1등석, 2=2등석, 3=3등석)
sex      : 성별
age      : 나이
sibsp    : 형제·배우자 탑승 수
parch    : 부모·자녀 탑승 수
fare     : 요금
embarked : 탑승 항구 (C=셰르부르, Q=퀸스타운, S=사우샘프턴)
""")
titanic_df.info()

In [ ]:
cols = titanic_df.columns[0:8]
df = titanic_df[cols]
df.info()

In [ ]:
df.describe()

In [ ]:
# df.info() # 결측값 확인
# df.isnull().sum() # 결측값 확인

# age: 중앙값 대체 (이상값의 영향을 적게 받는 통계량 사용)
df['age'] = df['age'].fillna(df['age'].median())

# embarked: 최빈값 대체
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

print('처리 후 결측치:')
print(df.isnull().sum())

In [ ]:
df.head()

In [ ]:
# 범주형 데이터 (문자열) -> 숫자형으로 변경

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import pandas as pd

# 원본 데이터셋 복제
df_enc = df.copy()

le = LabelEncoder() # 범주형 -> 숫자형 변경 도구
df_enc['sex'] = le.fit_transform(df_enc['sex'])     # female=0, male=1

df_enc = pd.get_dummies(df_enc, columns=['embarked'],
                        prefix='port', dtype=int) # OneHotEncoding 처리

print('인코딩 후 컬럼:')
print(df_enc.columns.tolist())
df_enc.head()

In [ ]:
# 숫자형 변수 스케일 조정 (각 변수의 값 범위를 동일하게 만드는 조정 )

from sklearn.preprocessing import StandardScaler, MinMaxScaler

scale_cols = ['age', 'fare']
#scaler = StandardScaler() # 입력 값을 평균 0, 표준편차 1의 분포로 변경
scaler = MinMaxScaler() # 입력 값을 0~1의 값으로 변경
df_enc[scale_cols] = scaler.fit_transform(df[scale_cols])

print('정규화 후 통계 (평균≈0, 표준편차≈1):')
print(df_enc[scale_cols].describe().round(3))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["font.family"] = "Malgun Gothic"  # 한글 폰트

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# 생존 여부
df['survived'].value_counts().plot(kind='bar', ax=axes[0,0],
    color=['salmon','steelblue'], edgecolor='white')
axes[0,0].set_title('생존 여부')
axes[0,0].set_xticklabels(['사망(0)','생존(1)'], rotation=0)

# # 성별 생존율
# df.groupby('sex')['survived'].mean().plot(kind='bar', ax=axes[0,1],
#     color=['steelblue','salmon'], edgecolor='white')
# axes[0,1].set_title('성별 생존율')
# axes[0,1].set_xticklabels(['남성','여성'], rotation=0)

# # 객실 등급별 생존율
# df.groupby('pclass')['survived'].mean().plot(kind='bar', ax=axes[1,0],
#     color=['gold','silver','peru'], edgecolor='white')
# axes[1,0].set_title('객실 등급별 생존율')
# axes[1,0].set_xticklabels(['1등석','2등석','3등석'], rotation=0)


# # 나이 분포 (생존 여부별)
# df[df['survived']==1]['age'].hist(bins=20, ax=axes[1,1], alpha=0.6,
#                                    label='생존', color='steelblue')
# df[df['survived']==0]['age'].hist(bins=20, ax=axes[1,1], alpha=0.6,
#                                    label='사망', color='salmon')
# axes[1,1].set_title('나이 분포 (생존 여부별)')
# axes[1,1].legend()

plt.suptitle('타이타닉 EDA', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eda_titanic.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# 데이터프레임을 파일에 저장
df_enc.to_csv('data-files/data2.csv', index=False)

In [ ]:
df_enc.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_cols = df_enc.columns[1:]
y_col = df_enc.columns[0]

X_train, X_test, y_train, y_test = train_test_split(df_enc[X_cols], df_enc[y_col],test_size=0.2, random_state=42, stratify=df_enc[y_col])

In [ ]:
import numpy as np
print(X_train.shape, X_test.shape)
np.unique(y_train,return_counts=True), np.unique(y_test, return_counts=True)

In [ ]:
X_train.to_csv('data-files/X_train.csv', index=False)
y_train.to_csv('data-files/y_train.csv', index=False)
X_test.to_csv('data-files/X_test.csv', index=False)
y_test.to_csv('data-files/y_test.csv', index=False)